# F5 Cross-Sell & Upsell Recommendation Model

Train a binary classifier on historical expansion deals (won vs lost) to predict which products each account is most likely to purchase next. Generates ranked, multi-product recommendations per account.

In [ ]:
# Setup and Connection
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE ROLE SYSADMIN").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()
session.sql("USE DATABASE F5_PROD").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS F5_PROD.FINAL").collect()

print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {session.get_current_database()}")

## Prepare Training Labels
Use historical expansion deals as ground truth. Positive = deal closed won. Negative = deal closed lost.

In [ ]:
# Prepare Training Labels from Expansion Deals
labels_sql = """
CREATE OR REPLACE TABLE F5_PROD.FINAL.CROSS_SELL_LABELS AS
SELECT DISTINCT
    li.SFDCF5_ACCT_ID,
    li.PRODUCT_SKU_ID AS CANDIDATE_SKU,
    CASE WHEN o.OPPORTUNITY_STAGE_NAME = 'Closed Won' THEN 1 ELSE 0 END AS TARGET
FROM F5_PROD.RAW.DIM_SALES_OPPORTUNITY o
JOIN F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM li 
    ON o.OPPORTUNITY_ID = li.OPPORTUNITY_ID
WHERE o.OPPORTUNITY_TYPE_CODE = 'Expansion'
  AND o.OPPORTUNITY_STAGE_NAME IN ('Closed Won', 'Closed Lost')
"""
session.sql(labels_sql).collect()

labels_df = session.table("F5_PROD.FINAL.CROSS_SELL_LABELS")
print(f"Training labels: {labels_df.count()} rows")
print(f"\nTarget distribution:")
labels_df.group_by("TARGET").count().show()
print(f"\nAccounts with labels: {labels_df.select('SFDCF5_ACCT_ID').distinct().count()}")
print(f"Products in expansion deals: {labels_df.select('CANDIDATE_SKU').distinct().count()}")

## Feature Engineering
Build features for each (account, candidate_product) pair: account characteristics, product attributes, and interaction signals.

In [ ]:
# Feature Engineering for (account, product) pairs
features_sql = """
CREATE OR REPLACE TABLE F5_PROD.FINAL.CROSS_SELL_FEATURES AS
WITH account_products AS (
    -- What each account currently owns
    SELECT SFDCF5_ACCT_ID, PRODUCT_SKU_ID
    FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM
    GROUP BY 1, 2
),
account_stats AS (
    -- Account-level features
    SELECT 
        a.SFDCF5_ACCT_ID,
        COUNT(DISTINCT ap.PRODUCT_SKU_ID) AS PRODUCTS_OWNED,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'BIG-IP' THEN ap.PRODUCT_SKU_ID END) AS BIGIP_COUNT,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'F5 XC' THEN ap.PRODUCT_SKU_ID END) AS XC_COUNT,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'NGINX' THEN ap.PRODUCT_SKU_ID END) AS NGINX_COUNT,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'Calypso' THEN ap.PRODUCT_SKU_ID END) AS AI_COUNT,
        COALESCE(SUM(li.TOTAL_PRICE_AMT), 0) AS TOTAL_SPEND
    FROM F5_PROD.RAW.DIM_CUST_ACCT_SFDC a
    LEFT JOIN account_products ap ON a.SFDCF5_ACCT_ID = ap.SFDCF5_ACCT_ID
    LEFT JOIN F5_PROD.RAW.DIM_PRODUCT_OFFER p ON ap.PRODUCT_SKU_ID = p.OFFER_SKU_ID
    LEFT JOIN F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM li ON a.SFDCF5_ACCT_ID = li.SFDCF5_ACCT_ID AND ap.PRODUCT_SKU_ID = li.PRODUCT_SKU_ID
    GROUP BY 1
),
telemetry_features AS (
    -- Telemetry signals per account
    SELECT SFDCF5_ACCT_ID,
        AVG(ACTIVE_HTTP_LOAD_BALANCER_QTY) AS AVG_HTTP_LB,
        AVG(WAF_USAGE_QTY) AS AVG_WAF,
        AVG(BOT_ADVANCED_TRANSACTION_CNT) AS AVG_BOT_TXN,
        AVG(ACTIVE_ENDPOINT_QTY) AS AVG_ENDPOINTS,
        AVG(DNS_ZONES_QTY) AS AVG_DNS,
        CASE
            WHEN AVG(BOT_ADVANCED_TRANSACTION_CNT) > 300000 THEN 'bot_defense'
            WHEN AVG(WAF_USAGE_QTY) > 25 THEN 'waf'
            WHEN AVG(ACTIVE_ENDPOINT_QTY) > 150 THEN 'capacity'
            WHEN AVG(ACTIVE_HTTP_LOAD_BALANCER_QTY) > 40 THEN 'load_balancer'
            WHEN AVG(DNS_ZONES_QTY) > 7 THEN 'dns'
            ELSE 'load_balancer'
        END AS DOMINANT_SIGNAL
    FROM F5_PROD.RAW.COL_XC_TELEMETRY
    WHERE OBSERVATION_DATE >= CURRENT_DATE() - 60
    GROUP BY 1
),
support_features AS (
    -- Support patterns per account
    SELECT SFDCF5_ACCT_ID,
        COUNT(*) AS CASE_COUNT_180D,
        COUNT(CASE WHEN CURRENT_PRIORITY_CODE IN ('P1 - Critical','P2 - High') THEN 1 END) AS HIGH_SEV_CASES
    FROM F5_PROD.RAW.DIM_SUPPORT_CASE
    WHERE CREATED_DATETIME >= DATEADD(day, -180, CURRENT_TIMESTAMP())
    GROUP BY 1
),
utilization_features AS (
    -- Max utilization and contract timing per account
    SELECT SALES_SFDCF5_ACCT_ID AS SFDCF5_ACCT_ID,
        MAX(FEATURE_USED_QTY / NULLIF(FEATURE_ENTITLED_QTY, 0)) AS MAX_UTILIZATION_PCT,
        MIN(MONTHS_LEFT_IN_TERM_NUM) AS MIN_MONTHS_LEFT
    FROM F5_PROD.RAW.COL_TERM_SUB_MONTHLY_USAGE_V2
    WHERE BILLING_MONTH_START_DATE = (SELECT MAX(BILLING_MONTH_START_DATE) FROM F5_PROD.RAW.COL_TERM_SUB_MONTHLY_USAGE_V2)
    GROUP BY 1
),
product_popularity AS (
    -- What % of accounts own each product
    SELECT PRODUCT_SKU_ID,
        COUNT(DISTINCT SFDCF5_ACCT_ID)::FLOAT / (SELECT COUNT(DISTINCT SFDCF5_ACCT_ID) FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM) AS POPULARITY_PCT
    FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM
    GROUP BY 1
),
co_purchase AS (
    -- Co-purchase frequency: for each product pair, how often bought together
    SELECT a.PRODUCT_SKU_ID AS OWNED_SKU, b.PRODUCT_SKU_ID AS CANDIDATE_SKU,
        COUNT(DISTINCT a.SFDCF5_ACCT_ID)::FLOAT / 
            NULLIF((SELECT COUNT(DISTINCT SFDCF5_ACCT_ID) FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM WHERE PRODUCT_SKU_ID = a.PRODUCT_SKU_ID), 0) AS CO_PURCHASE_RATE
    FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM a
    JOIN F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM b 
        ON a.SFDCF5_ACCT_ID = b.SFDCF5_ACCT_ID AND a.PRODUCT_SKU_ID != b.PRODUCT_SKU_ID
    GROUP BY 1, 2
)
-- Join labels with features
SELECT 
    l.SFDCF5_ACCT_ID,
    l.CANDIDATE_SKU,
    l.TARGET,
    -- Account features
    COALESCE(ast.PRODUCTS_OWNED, 0) AS PRODUCTS_OWNED,
    COALESCE(ast.BIGIP_COUNT, 0) AS BIGIP_COUNT,
    COALESCE(ast.XC_COUNT, 0) AS XC_COUNT,
    COALESCE(ast.NGINX_COUNT, 0) AS NGINX_COUNT,
    COALESCE(ast.AI_COUNT, 0) AS AI_COUNT,
    COALESCE(ast.TOTAL_SPEND, 0) AS TOTAL_SPEND,
    -- Telemetry features
    COALESCE(tf.AVG_HTTP_LB, 0) AS AVG_HTTP_LB,
    COALESCE(tf.AVG_WAF, 0) AS AVG_WAF,
    COALESCE(tf.AVG_BOT_TXN, 0) AS AVG_BOT_TXN,
    COALESCE(tf.AVG_ENDPOINTS, 0) AS AVG_ENDPOINTS,
    COALESCE(tf.AVG_DNS, 0) AS AVG_DNS,
    COALESCE(tf.DOMINANT_SIGNAL, 'unknown') AS DOMINANT_SIGNAL,
    -- Support features
    COALESCE(sf.CASE_COUNT_180D, 0) AS CASE_COUNT_180D,
    COALESCE(sf.HIGH_SEV_CASES, 0) AS HIGH_SEV_CASES,
    -- Utilization features
    COALESCE(uf.MAX_UTILIZATION_PCT, 0) AS MAX_UTILIZATION_PCT,
    COALESCE(uf.MIN_MONTHS_LEFT, 12) AS MIN_MONTHS_LEFT,
    -- Product features
    p.CORE_PRODUCT_NAME AS CANDIDATE_PRODUCT_FAMILY,
    p.PRODUCT_LINE_TYPE AS CANDIDATE_PRODUCT_TYPE,
    COALESCE(p.STANDARD_LIST_PRICE_AMT, 0) AS CANDIDATE_LIST_PRICE,
    COALESCE(pp.POPULARITY_PCT, 0) AS CANDIDATE_POPULARITY,
    -- Interaction: does account own other products in same family?
    CASE WHEN p.CORE_PRODUCT_NAME = 'BIG-IP' AND ast.BIGIP_COUNT > 0 THEN 1
         WHEN p.CORE_PRODUCT_NAME = 'F5 XC' AND ast.XC_COUNT > 0 THEN 1
         WHEN p.CORE_PRODUCT_NAME = 'NGINX' AND ast.NGINX_COUNT > 0 THEN 1
         WHEN p.CORE_PRODUCT_NAME = 'Calypso' AND ast.AI_COUNT > 0 THEN 1
         ELSE 0 END AS OWNS_SAME_FAMILY,
    -- Co-purchase score: avg co-purchase rate between candidate and account's owned products
    COALESCE(cp_avg.AVG_CO_PURCHASE, 0) AS AVG_CO_PURCHASE_SCORE
FROM F5_PROD.FINAL.CROSS_SELL_LABELS l
LEFT JOIN account_stats ast ON l.SFDCF5_ACCT_ID = ast.SFDCF5_ACCT_ID
LEFT JOIN telemetry_features tf ON l.SFDCF5_ACCT_ID = tf.SFDCF5_ACCT_ID
LEFT JOIN support_features sf ON l.SFDCF5_ACCT_ID = sf.SFDCF5_ACCT_ID
LEFT JOIN utilization_features uf ON l.SFDCF5_ACCT_ID = uf.SFDCF5_ACCT_ID
LEFT JOIN F5_PROD.RAW.DIM_PRODUCT_OFFER p ON l.CANDIDATE_SKU = p.OFFER_SKU_ID
LEFT JOIN product_popularity pp ON l.CANDIDATE_SKU = pp.PRODUCT_SKU_ID
LEFT JOIN (
    -- Average co-purchase score between candidate and all products account owns
    SELECT ap.SFDCF5_ACCT_ID, cp.CANDIDATE_SKU, AVG(cp.CO_PURCHASE_RATE) AS AVG_CO_PURCHASE
    FROM account_products ap
    JOIN co_purchase cp ON ap.PRODUCT_SKU_ID = cp.OWNED_SKU
    GROUP BY 1, 2
) cp_avg ON l.SFDCF5_ACCT_ID = cp_avg.SFDCF5_ACCT_ID AND l.CANDIDATE_SKU = cp_avg.CANDIDATE_SKU
"""

session.sql(features_sql).collect()

features_df = session.table("F5_PROD.FINAL.CROSS_SELL_FEATURES")
print(f"Feature table: {features_df.count()} rows, {len(features_df.columns)} columns")
print(f"\nColumns: {features_df.columns}")
print(f"\nSample:")
features_df.limit(5).show()

## Feature Store Registration
Register features for centralized management and reproducible inference.

In [ ]:
# Feature Store Registration
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database="F5_PROD",
    name="FINAL",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

# Composite entity: account + product pair
acct_product_entity = Entity(
    name="ACCOUNT_PRODUCT",
    join_keys=["SFDCF5_ACCT_ID", "CANDIDATE_SKU"],
    desc="Account-product pair for cross-sell prediction"
)
fs.register_entity(acct_product_entity)

# Register feature view (exclude target)
feature_df = session.table("F5_PROD.FINAL.CROSS_SELL_FEATURES").drop("TARGET")

# Drop existing if re-running
session.sql("DROP TABLE IF EXISTS F5_PROD.FINAL.CROSS_SELL_TRAINING_DS_V1").collect()

cross_sell_fv = FeatureView(
    name="FV_CROSS_SELL",
    entities=[acct_product_entity],
    feature_df=feature_df,
    desc="Cross-sell features: account stats, telemetry, support, utilization, product attributes, co-purchase"
)
cross_sell_fv = fs.register_feature_view(cross_sell_fv, version="V1", block=True)
print(f"Feature View registered: {cross_sell_fv.name} v{cross_sell_fv.version}")
print(f"Features: {len(cross_sell_fv.feature_names)} columns")

# Generate training dataset
spine_df = session.table("F5_PROD.FINAL.CROSS_SELL_FEATURES").select(
    "SFDCF5_ACCT_ID", "CANDIDATE_SKU", "TARGET"
)

dataset = fs.generate_dataset(
    name="F5_PROD.FINAL.CROSS_SELL_TRAINING_DS",
    spine_df=spine_df,
    features=[cross_sell_fv],
    version="V1",
    output_type="table",
    spine_label_cols=["TARGET"],
    desc="Training dataset for cross-sell model"
)

training_df = session.table("F5_PROD.FINAL.CROSS_SELL_TRAINING_DS_V1")
print(f"\nTraining dataset: {training_df.count()} rows")

## Model Training
XGBoost binary classifier with GridSearch. Predicts probability that an account will purchase a given product.

In [ ]:
# Model Training - Local sklearn (avoids warehouse numpy conflict)
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from xgboost import XGBClassifier

# Load training data to pandas (small dataset, trains locally in container)
train_data = session.table("F5_PROD.FINAL.CROSS_SELL_TRAINING_DS_V1")
df = train_data.to_pandas()

# Column setup
label_col = "TARGET"
id_cols = ["SFDCF5_ACCT_ID", "CANDIDATE_SKU"]
cat_cols = ["DOMINANT_SIGNAL", "CANDIDATE_PRODUCT_FAMILY", "CANDIDATE_PRODUCT_TYPE"]
num_cols = [c for c in df.columns if c not in cat_cols + id_cols + [label_col]]

print(f"Total rows: {len(df)}")
print(f"Categorical: {cat_cols}")
print(f"Numeric ({len(num_cols)}): {num_cols}")

# Preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", MinMaxScaler(clip=True), num_cols)
])

# Split
X = df[cat_cols + num_cols]
y = df[label_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Class weight
n_pos = int(y.sum())
n_neg = int(len(y) - n_pos)
class_weight = n_neg / n_pos
print(f"\nClass balance: {n_pos} positive, {n_neg} negative (weight: {class_weight:.2f})")
print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")

# Build pipeline
pipeline = SkPipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(scale_pos_weight=class_weight, random_state=42, eval_metric="logloss"))
])

# GridSearch
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [3, 4, 5],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1", n_jobs=1)
print("\nTraining with GridSearch...")
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Best params: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

# Store for later cells
FEATURE_COLS = cat_cols + num_cols

## Model Evaluation
Classification metrics on hold-out test set plus recommendation quality checks.

In [ ]:
# Model Evaluation
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score

# Predict on test set
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "auc_roc": roc_auc_score(y_test, y_proba),
}

print("=" * 50)
print("CLASSIFICATION METRICS (Hold-out Test Set)")
print("=" * 50)
for metric, value in metrics.items():
    print(f"  {metric:12s}: {value:.4f}")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  TN={cm[0][0]:3d}  FP={cm[0][1]:3d}")
print(f"  FN={cm[1][0]:3d}  TP={cm[1][1]:3d}")

# Feature importance from the XGBoost step of the pipeline
xgb_model = best_model.named_steps["classifier"]
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
importances = xgb_model.feature_importances_

feat_imp = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)[:10]
print("\n" + "=" * 50)
print("TOP 10 FEATURES DRIVING RECOMMENDATIONS")
print("=" * 50)
for name, imp in feat_imp:
    bar = "█" * int(imp * 40)
    print(f"  {name:35s} {imp:.4f} {bar}")

## Model Registry
Log the trained model with metrics for versioning and deployment.

In [ ]:
# Register in Model Registry
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="F5_PROD", schema_name="FINAL")

model_name = "CROSS_SELL_XGB_MODEL"
version_name = "V1"

# Log the sklearn pipeline (includes preprocessor + XGBoost)
mv = reg.log_model(
    model_name=model_name,
    version_name=version_name,
    model=best_model,
    metrics=metrics,
    sample_input_data=X_train.head(100),
    conda_dependencies=["scikit-learn", "xgboost", "numpy"]
)

mv.comment = "XGBoost cross-sell model trained on expansion deal outcomes. Predicts purchase probability for account-product pairs."

print(f"Model registered: {model_name} (version {version_name})")
print(f"Metrics: {metrics}")
print(f"\nRegistered models:")
print(reg.show_models())

## Batch Inference - Generate Recommendations
Score all (account, product) pairs where account doesn't already own the product. Rank by probability and write top recommendations per account.

In [ ]:
# Batch Inference - Score all account-product pairs
# Generate inference set (all account-product pairs not yet owned)
inference_sql = """
CREATE OR REPLACE TABLE F5_PROD.FINAL.CROSS_SELL_INFERENCE_SET AS
WITH account_products AS (
    SELECT SFDCF5_ACCT_ID, PRODUCT_SKU_ID
    FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM
    GROUP BY 1, 2
),
all_accounts AS (
    SELECT DISTINCT SFDCF5_ACCT_ID FROM F5_PROD.RAW.DIM_CUST_ACCT_SFDC
),
all_products AS (
    SELECT OFFER_SKU_ID FROM F5_PROD.RAW.DIM_PRODUCT_OFFER
    WHERE OFFER_SKU_ID NOT IN ('F5-PS-IMPLEMENTATION','F5-PS-TRAINING','F5-SUP-PREMIUM','F5-SUP-STANDARD')
),
candidate_pairs AS (
    SELECT a.SFDCF5_ACCT_ID, p.OFFER_SKU_ID AS CANDIDATE_SKU
    FROM all_accounts a
    CROSS JOIN all_products p
    WHERE NOT EXISTS (
        SELECT 1 FROM account_products ap 
        WHERE ap.SFDCF5_ACCT_ID = a.SFDCF5_ACCT_ID AND ap.PRODUCT_SKU_ID = p.OFFER_SKU_ID
    )
),
account_stats AS (
    SELECT a.SFDCF5_ACCT_ID,
        COUNT(DISTINCT ap.PRODUCT_SKU_ID) AS PRODUCTS_OWNED,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'BIG-IP' THEN ap.PRODUCT_SKU_ID END) AS BIGIP_COUNT,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'F5 XC' THEN ap.PRODUCT_SKU_ID END) AS XC_COUNT,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'NGINX' THEN ap.PRODUCT_SKU_ID END) AS NGINX_COUNT,
        COUNT(DISTINCT CASE WHEN p.CORE_PRODUCT_NAME = 'Calypso' THEN ap.PRODUCT_SKU_ID END) AS AI_COUNT,
        COALESCE(SUM(li.TOTAL_PRICE_AMT), 0) AS TOTAL_SPEND
    FROM F5_PROD.RAW.DIM_CUST_ACCT_SFDC a
    LEFT JOIN account_products ap ON a.SFDCF5_ACCT_ID = ap.SFDCF5_ACCT_ID
    LEFT JOIN F5_PROD.RAW.DIM_PRODUCT_OFFER p ON ap.PRODUCT_SKU_ID = p.OFFER_SKU_ID
    LEFT JOIN F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM li ON a.SFDCF5_ACCT_ID = li.SFDCF5_ACCT_ID AND ap.PRODUCT_SKU_ID = li.PRODUCT_SKU_ID
    GROUP BY 1
),
telemetry_features AS (
    SELECT SFDCF5_ACCT_ID,
        AVG(ACTIVE_HTTP_LOAD_BALANCER_QTY) AS AVG_HTTP_LB,
        AVG(WAF_USAGE_QTY) AS AVG_WAF,
        AVG(BOT_ADVANCED_TRANSACTION_CNT) AS AVG_BOT_TXN,
        AVG(ACTIVE_ENDPOINT_QTY) AS AVG_ENDPOINTS,
        AVG(DNS_ZONES_QTY) AS AVG_DNS,
        CASE
            WHEN AVG(BOT_ADVANCED_TRANSACTION_CNT) > 300000 THEN 'bot_defense'
            WHEN AVG(WAF_USAGE_QTY) > 25 THEN 'waf'
            WHEN AVG(ACTIVE_ENDPOINT_QTY) > 150 THEN 'capacity'
            WHEN AVG(ACTIVE_HTTP_LOAD_BALANCER_QTY) > 40 THEN 'load_balancer'
            WHEN AVG(DNS_ZONES_QTY) > 7 THEN 'dns'
            ELSE 'load_balancer'
        END AS DOMINANT_SIGNAL
    FROM F5_PROD.RAW.COL_XC_TELEMETRY
    WHERE OBSERVATION_DATE >= CURRENT_DATE() - 60
    GROUP BY 1
),
support_features AS (
    SELECT SFDCF5_ACCT_ID,
        COUNT(*) AS CASE_COUNT_180D,
        COUNT(CASE WHEN CURRENT_PRIORITY_CODE IN ('P1 - Critical','P2 - High') THEN 1 END) AS HIGH_SEV_CASES
    FROM F5_PROD.RAW.DIM_SUPPORT_CASE
    WHERE CREATED_DATETIME >= DATEADD(day, -180, CURRENT_TIMESTAMP())
    GROUP BY 1
),
utilization_features AS (
    SELECT SALES_SFDCF5_ACCT_ID AS SFDCF5_ACCT_ID,
        MAX(FEATURE_USED_QTY / NULLIF(FEATURE_ENTITLED_QTY, 0)) AS MAX_UTILIZATION_PCT,
        MIN(MONTHS_LEFT_IN_TERM_NUM) AS MIN_MONTHS_LEFT
    FROM F5_PROD.RAW.COL_TERM_SUB_MONTHLY_USAGE_V2
    WHERE BILLING_MONTH_START_DATE = (SELECT MAX(BILLING_MONTH_START_DATE) FROM F5_PROD.RAW.COL_TERM_SUB_MONTHLY_USAGE_V2)
    GROUP BY 1
),
product_popularity AS (
    SELECT PRODUCT_SKU_ID,
        COUNT(DISTINCT SFDCF5_ACCT_ID)::FLOAT / (SELECT COUNT(DISTINCT SFDCF5_ACCT_ID) FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM) AS POPULARITY_PCT
    FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM
    GROUP BY 1
),
co_purchase AS (
    SELECT a.PRODUCT_SKU_ID AS OWNED_SKU, b.PRODUCT_SKU_ID AS CANDIDATE_SKU,
        COUNT(DISTINCT a.SFDCF5_ACCT_ID)::FLOAT / 
            NULLIF((SELECT COUNT(DISTINCT SFDCF5_ACCT_ID) FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM WHERE PRODUCT_SKU_ID = a.PRODUCT_SKU_ID), 0) AS CO_PURCHASE_RATE
    FROM F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM a
    JOIN F5_PROD.RAW.COL_SALES_OPPORTUNITY_LINE_ITEM b 
        ON a.SFDCF5_ACCT_ID = b.SFDCF5_ACCT_ID AND a.PRODUCT_SKU_ID != b.PRODUCT_SKU_ID
    GROUP BY 1, 2
)
SELECT 
    cp.SFDCF5_ACCT_ID,
    cp.CANDIDATE_SKU,
    COALESCE(ast.PRODUCTS_OWNED, 0) AS PRODUCTS_OWNED,
    COALESCE(ast.BIGIP_COUNT, 0) AS BIGIP_COUNT,
    COALESCE(ast.XC_COUNT, 0) AS XC_COUNT,
    COALESCE(ast.NGINX_COUNT, 0) AS NGINX_COUNT,
    COALESCE(ast.AI_COUNT, 0) AS AI_COUNT,
    COALESCE(ast.TOTAL_SPEND, 0) AS TOTAL_SPEND,
    COALESCE(tf.AVG_HTTP_LB, 0) AS AVG_HTTP_LB,
    COALESCE(tf.AVG_WAF, 0) AS AVG_WAF,
    COALESCE(tf.AVG_BOT_TXN, 0) AS AVG_BOT_TXN,
    COALESCE(tf.AVG_ENDPOINTS, 0) AS AVG_ENDPOINTS,
    COALESCE(tf.AVG_DNS, 0) AS AVG_DNS,
    COALESCE(tf.DOMINANT_SIGNAL, 'unknown') AS DOMINANT_SIGNAL,
    COALESCE(sf.CASE_COUNT_180D, 0) AS CASE_COUNT_180D,
    COALESCE(sf.HIGH_SEV_CASES, 0) AS HIGH_SEV_CASES,
    COALESCE(uf.MAX_UTILIZATION_PCT, 0) AS MAX_UTILIZATION_PCT,
    COALESCE(uf.MIN_MONTHS_LEFT, 12) AS MIN_MONTHS_LEFT,
    p.CORE_PRODUCT_NAME AS CANDIDATE_PRODUCT_FAMILY,
    p.PRODUCT_LINE_TYPE AS CANDIDATE_PRODUCT_TYPE,
    COALESCE(p.STANDARD_LIST_PRICE_AMT, 0) AS CANDIDATE_LIST_PRICE,
    COALESCE(pp.POPULARITY_PCT, 0) AS CANDIDATE_POPULARITY,
    CASE WHEN p.CORE_PRODUCT_NAME = 'BIG-IP' AND ast.BIGIP_COUNT > 0 THEN 1
         WHEN p.CORE_PRODUCT_NAME = 'F5 XC' AND ast.XC_COUNT > 0 THEN 1
         WHEN p.CORE_PRODUCT_NAME = 'NGINX' AND ast.NGINX_COUNT > 0 THEN 1
         WHEN p.CORE_PRODUCT_NAME = 'Calypso' AND ast.AI_COUNT > 0 THEN 1
         ELSE 0 END AS OWNS_SAME_FAMILY,
    COALESCE(cp_avg.AVG_CO_PURCHASE, 0) AS AVG_CO_PURCHASE_SCORE
FROM candidate_pairs cp
LEFT JOIN account_stats ast ON cp.SFDCF5_ACCT_ID = ast.SFDCF5_ACCT_ID
LEFT JOIN telemetry_features tf ON cp.SFDCF5_ACCT_ID = tf.SFDCF5_ACCT_ID
LEFT JOIN support_features sf ON cp.SFDCF5_ACCT_ID = sf.SFDCF5_ACCT_ID
LEFT JOIN utilization_features uf ON cp.SFDCF5_ACCT_ID = uf.SFDCF5_ACCT_ID
LEFT JOIN F5_PROD.RAW.DIM_PRODUCT_OFFER p ON cp.CANDIDATE_SKU = p.OFFER_SKU_ID
LEFT JOIN product_popularity pp ON cp.CANDIDATE_SKU = pp.PRODUCT_SKU_ID
LEFT JOIN (
    SELECT ap2.SFDCF5_ACCT_ID, cpr.CANDIDATE_SKU, AVG(cpr.CO_PURCHASE_RATE) AS AVG_CO_PURCHASE
    FROM account_products ap2
    JOIN co_purchase cpr ON ap2.PRODUCT_SKU_ID = cpr.OWNED_SKU
    GROUP BY 1, 2
) cp_avg ON cp.SFDCF5_ACCT_ID = cp_avg.SFDCF5_ACCT_ID AND cp.CANDIDATE_SKU = cp_avg.CANDIDATE_SKU
"""

print("Building inference set...")
session.sql(inference_sql).collect()

# Load to pandas and score locally
inf_df = session.table("F5_PROD.FINAL.CROSS_SELL_INFERENCE_SET").to_pandas()
print(f"Inference set: {len(inf_df)} (account, product) pairs to score")

# Score using the trained pipeline
X_inf = inf_df[cat_cols + num_cols]
inf_df["CONFIDENCE_SCORE"] = best_model.predict_proba(X_inf)[:, 1]

# Rank within each account and take top 5
inf_df["PRIORITY_RANK"] = inf_df.groupby("SFDCF5_ACCT_ID")["CONFIDENCE_SCORE"].rank(ascending=False, method="first").astype(int)
recs = inf_df[inf_df["PRIORITY_RANK"] <= 5].copy()

# Get account names
acct_names = session.sql("SELECT SFDCF5_ACCT_ID, ACCT_NAME FROM F5_PROD.RAW.DIM_CUST_ACCT_SFDC").to_pandas()
recs = recs.merge(acct_names, on="SFDCF5_ACCT_ID", how="left")

# Determine recommendation type
def get_rec_type(row):
    if row["OWNS_SAME_FAMILY"] == 0:
        return "cross-sell"
    elif row["MAX_UTILIZATION_PCT"] > 0.8:
        return "capacity"
    else:
        return "upsell"

recs["RECOMMENDATION_TYPE"] = recs.apply(get_rec_type, axis=1)

# Generate rationale with friendly feature names and product descriptions
feature_friendly_names = {
    "num__AVG_HTTP_LB": "load balancer usage",
    "num__AVG_WAF": "WAF activity",
    "num__AVG_BOT_TXN": "bot traffic volume",
    "num__AVG_ENDPOINTS": "endpoint count",
    "num__AVG_DNS": "DNS zone count",
    "num__PRODUCTS_OWNED": "product portfolio size",
    "num__BIGIP_COUNT": "BIG-IP product count",
    "num__XC_COUNT": "XC product count",
    "num__NGINX_COUNT": "NGINX product count",
    "num__AI_COUNT": "AI security product count",
    "num__TOTAL_SPEND": "total account spend",
    "num__CASE_COUNT_180D": "recent support cases",
    "num__HIGH_SEV_CASES": "high severity cases",
    "num__MAX_UTILIZATION_PCT": "high capacity utilization",
    "num__MIN_MONTHS_LEFT": "upcoming renewal",
    "num__CANDIDATE_LIST_PRICE": "product price point",
    "num__CANDIDATE_POPULARITY": "product popularity",
    "num__OWNS_SAME_FAMILY": "existing family ownership",
    "num__AVG_CO_PURCHASE_SCORE": "co-purchase affinity",
}

top_feature_names = [f[0] for f in feat_imp[:3]]
friendly_top = [feature_friendly_names.get(f, f.replace("num__", "").replace("cat__", "").replace("_", " ").lower()) for f in top_feature_names]

# Get product descriptions for richer rationale
product_descs = session.sql("SELECT OFFER_SKU_ID, OFFER_DESC, F5_PRODUCT_OFFER_FAMILY_NAME FROM F5_PROD.RAW.DIM_PRODUCT_OFFER").to_pandas()
recs = recs.merge(product_descs, left_on="CANDIDATE_SKU", right_on="OFFER_SKU_ID", how="left")

recs["RATIONALE"] = recs.apply(
    lambda r: f"{r['OFFER_DESC']} ({r['F5_PRODUCT_OFFER_FAMILY_NAME']}). "
              f"Confidence {r['CONFIDENCE_SCORE']:.0%} based on {', '.join(friendly_top)}. "
              f"{'New product family - expand into this category.' if r['RECOMMENDATION_TYPE'] == 'cross-sell' else 'Add capacity to existing product line.' if r['RECOMMENDATION_TYPE'] == 'capacity' else 'Upgrade within current product family.'} "
              f"List price: ${r['CANDIDATE_LIST_PRICE']:,.0f}.",
    axis=1
)

recs["PREDICTION_DATE"] = pd.Timestamp.now().date()

# Final output
final_recs = recs[[
    "SFDCF5_ACCT_ID", "ACCT_NAME", "CANDIDATE_SKU", "RECOMMENDATION_TYPE",
    "CONFIDENCE_SCORE", "RATIONALE", "PRIORITY_RANK", "PREDICTION_DATE"
]].rename(columns={"CANDIDATE_SKU": "RECOMMENDED_SKU"})

# Write to Snowflake
session.write_pandas(
    final_recs,
    table_name="CROSS_SELL_RECOMMENDATIONS",
    database="F5_PROD",
    schema="FINAL",
    overwrite=True,
    auto_create_table=True
)

# Verify
result_table = session.table("F5_PROD.FINAL.CROSS_SELL_RECOMMENDATIONS")
print(f"\nCROSS_SELL_RECOMMENDATIONS: {result_table.count()} total recommendations")
print(f"Accounts: {result_table.select('SFDCF5_ACCT_ID').distinct().count()}")
print(f"\nType distribution:")
result_table.group_by("RECOMMENDATION_TYPE").count().show()
print(f"\nSample - Top 3 for first 3 accounts:")
result_table.filter(F.col("PRIORITY_RANK") <= 3).order_by("ACCT_NAME", "PRIORITY_RANK").limit(9).show()